In [2]:
import pandas as pd
import mplfinance as mpf
import matplotlib
import os

matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
from PIL import Image
from joblib import Parallel, delayed
from tqdm import tqdm
print("Finished importing Libraries")
##############################
#       USER PARAMETERS      #
##############################

#CSV_PATH = r"CRSP_kun_3_aktier_test_data.csv"
CSV_PATH = os.path.abspath("CRSP_top100_latest.csv")
#CSV_PATH = os.path.abspath("CRSP_kun_3_aktier_test_data.csv")
print("Current working directory:", os.getcwd())
print("Absolute CSV path:", CSV_PATH)
print("Exists:", os.path.exists(CSV_PATH))

df = pd.read_csv(CSV_PATH)

# -----------------------------
# Add or remove elements from the generated images here
# -----------------------------

ma_bb_period = 20 # either 5 (hvad Supervisor og Jonas bruger) eller 20 (hvad litteraturen siger er mere normalt/forsvarligt at bruge)

USE_MA = True
MA_PERIOD = ma_bb_period

USE_BB = False         # <-- set True to compute BB columns
BB_PERIOD = ma_bb_period
BB_NUM_STD = 2

USE_VOLUME = True

USE_RSI = False   # <-- set True to compute + overlay RSI
RSI_PERIOD = 14

# Folder name for this exact image combination
combo_folder_name = (
    f"vol_{int(USE_VOLUME)}"
    f"_ma_{int(USE_MA)}"
    f"_bb_{int(USE_BB)}"
    f"_rsi_{int(USE_RSI)}"
)
# -----------------------------
# END OF
# -----------------------------

OUTPUT_DIR = combo_folder_name
os.makedirs(OUTPUT_DIR, exist_ok=True)
LABELS_CSV_PATH = os.path.join(OUTPUT_DIR, "labels.csv")

print("OUTPUT_DIR is:", OUTPUT_DIR)
print("Exists?", os.path.exists(OUTPUT_DIR))

WINDOW_SIZE = 5  # Number of days for the chart (and for computing the moving average)
HORIZON = 5       # How far in the future we try to predict (lasdt day in windows + "Horizon" days ahead)

START_DATE = pd.to_datetime("2020-01-02") # FIRST date of the datsetset is "2020-01-02"
END_DATE   = pd.to_datetime("2024-12-31") # LAST date of the datsetset is "2024-12-31"

TICKER_LIMIT = 11       # Set to an integer to process only a subset of tickers. SÆTTES TIL "NONE" VED UBEGRÆNSET
TICKER_START_INDEX = 0    # Skip tickers before this index for chunked processing

##############################
#       DATA PREPARATION     #
##############################


df['DlyCalDt'] = pd.to_datetime(df['DlyCalDt'], format='%Y-%m-%d')
#df = df.sort_values(by='DlyCalDt')
df.rename(columns={
    'DlyOpen': 'Open',
    'DlyHigh': 'High',
    'DlyLow': 'Low',
    'DlyClose': 'Close',
    'DlyVol': 'Volume',
    'DlyCalDt': 'Date'
}, inplace=True)

# Filter data to the desired date range
df = df[(df['Date'] >= START_DATE) & (df['Date'] <= END_DATE)].copy()

# Group the data by ticker
grouped = list(df.groupby('Ticker'))

# --- Determine warm-up (how many initial rows to skip) ---
warmup = 0

# -----------------------------
# RSI toggle + per-ticker compute (no cross-ticker leakage)
# -----------------------------

import numpy as np

def _compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

if USE_RSI:
    # Build a new grouped list where each ticker DataFrame includes an RSI column
    grouped_with_rsi = []
    for tkr, g in grouped:
        g = g.sort_values('Date')
        g['RSI'] = _compute_rsi(g['Close'], RSI_PERIOD)
        grouped_with_rsi.append((tkr, g))
    grouped = grouped_with_rsi

if USE_RSI:
    warmup = max(warmup, RSI_PERIOD)   # safe default
# -----------------------------
# END of RSI
# -----------------------------

# -----------------------------
# Bollinger Bands toggle + per-ticker compute (no cross-ticker leakage)
# -----------------------------


def _compute_bollinger_bands(close: pd.Series, period: int = 20, num_std: float = 2.0):
    ma = close.rolling(window=period, min_periods=period).mean()
    sd = close.rolling(window=period, min_periods=period).std(ddof=0)  # ddof=0 is common in BB implementations
    upper = ma + num_std * sd
    lower = ma - num_std * sd
    return ma, upper, lower

if USE_BB:
    grouped_with_bb = []
    for tkr, g in grouped:
        g = g.sort_values('Date')

        bb_mid, bb_upper, bb_lower = _compute_bollinger_bands(
            g['Close'], period=BB_PERIOD, num_std=BB_NUM_STD
        )

        g['BB_MID'] = bb_mid
        g['BB_UPPER'] = bb_upper
        g['BB_LOWER'] = bb_lower

        grouped_with_bb.append((tkr, g))

    grouped = grouped_with_bb

# Bollinger Bands warmup: first BB_PERIOD-1 rows are NaN
if USE_BB:
    warmup = max(warmup, BB_PERIOD - 1)

# -----------------------------
# END OF: Bollinger Bands
# -----------------------------

# -----------------------------
# Moving Average toggle + parameters
# -----------------------------

def _compute_moving_average(close: pd.Series, period: int = 20) -> pd.Series:
    return close.rolling(window=period, min_periods=period).mean()

if USE_MA:
    grouped_with_ma = []
    for tkr, g in grouped:
        g = g.sort_values('Date')

        g['MA'] = _compute_moving_average(g['Close'], MA_PERIOD)

        grouped_with_ma.append((tkr, g))

    grouped = grouped_with_ma

# Moving Average warmup: first MA_PERIOD-1 rows are NaN
if USE_MA:
    warmup = max(warmup, MA_PERIOD - 1)
# -----------------------------
# END OF: Moving Average
# -----------------------------


if TICKER_LIMIT is not None:
    grouped = grouped[:TICKER_LIMIT]
# Apply ticker start index to allow chunked processing
grouped = grouped[TICKER_START_INDEX:]
print(f"Processing {len(grouped)} tickers from {START_DATE.date()} to {END_DATE.date()}...")

##############################
#   HELPER FUNCTIONALITY     #
##############################

def create_mplfinance_chart(data_window, save_path, mav, xlim=None, add_rsi=False, add_bb=False, add_ma=False, add_volume=False):
    """
    Create a candlestick chart with volume and a moving average (period=mav)
    using the provided data window. The chart is generated on a 4x4 figure with axes turned off.
    The x-axis is limited to display only the current window (using xlim).
    After saving, the image is resized to 224x224 pixels.
    """
    width_config = {
        'candle_width': 0.8,
        'candle_linewidth':5.0,
        'volume_width': 0.8,
        'volume_linewidth': 0.5
    }

    addplots = []

    if add_volume and add_rsi:
        panel_ratios = (3, 1, 1)
        rsi_panel = 2
    elif add_volume and not add_rsi:
        panel_ratios = (3, 1)
        rsi_panel = None
    elif not add_volume and add_rsi:
        panel_ratios = (3, 1)
        rsi_panel = 1
    else:
        panel_ratios = (3,)
        rsi_panel = None

    # --- Moving Average on the main price panel (panel=0) ---
    if add_ma:
        addplots += [
            mpf.make_addplot(data_window["MA"], panel=0, color="blue"),
        ]


    # --- Bollinger Bands on the main price panel (panel=0) ---
    if add_bb:
        addplots += [
            mpf.make_addplot(data_window["BB_UPPER"], panel=0, color="white"),
            mpf.make_addplot(data_window["BB_MID"],   panel=0, color="blue"),
            mpf.make_addplot(data_window["BB_LOWER"], panel=0, color="yellow"),
        ]
    
    if add_rsi and 'RSI' in data_window.columns:
        addplots += [
            mpf.make_addplot(data_window['RSI'], panel=rsi_panel, ylabel='RSI', ylim=(0, 100), color="orange"),
        ]
    
    

    # mplfinance expects addplot=None if there are no addplots
    addplots = addplots if len(addplots) > 0 else None

    fig, _ = mpf.plot(
        data_window,
        type='candlestick',
        volume=add_volume,
        #mav=5,
        style='charles',
        axisoff=True,
        returnfig=True,
        figsize=(4, 4),
        update_width_config=width_config,
        addplot=addplots,
        panel_ratios=panel_ratios,
        xlim=(-0.5, 5 - 0.5)
    )
    
    # Gemmer figuren som billede og lukker den (så der ikke hober sig figurer op).
    fig.savefig(save_path, bbox_inches='tight', pad_inches=0, facecolor='black')
    plt.close(fig)
    
    # Resize the saved image to 224x224 pixels
    with Image.open(save_path) as img:
        img_resized = img.resize((224, 224), Image.Resampling.BILINEAR)
        img_resized.save(save_path)

def generate_images_for_ticker(ticker, ticker_data, window_size, horizon, output_dir):
    """
    "Horizon" is the variable that decides how far in the future the code try to predict.
    To compute the moving average correctly over the window period, extend the data backward by
    (window_size - 1) rows (if available). Images are saved in a subfolder under output_dir,
    and filenames include the ticker and the date range.
    """
    ticker_data = ticker_data.copy()

    if "Date" in ticker_data.columns:
        ticker_data = ticker_data.sort_values("Date").set_index("Date")
    else:
        ticker_data = ticker_data.sort_index()

    ticker_data["future_close"] = ticker_data["Close"].shift(-horizon)
    ticker_data = ticker_data.dropna(subset=["future_close"]).copy()
    ticker_data["label"] = (ticker_data["future_close"] > ticker_data["Close"]).astype(int)

    n = len(ticker_data)
    label_rows = []
    
    # Create a subfolder for the ticker.
    ticker_folder = os.path.join(output_dir, str(ticker)) # default
    os.makedirs(ticker_folder, exist_ok=True) # default

    #Nedestående burde slette tidligere billeder, så du er sikker på dem der kommer nu er nye
    for f in os.listdir(ticker_folder):
        if f.endswith(".png"):
            os.remove(os.path.join(ticker_folder, f))
    
    window_index = 0
    # Loop using non-overlapping windows with step = window_size
    for i in range(warmup, n - window_size + 1, window_size):
        # Get the current window for display.
        data_to_plot = ticker_data.iloc[i : i + window_size]

        anchor_row = data_to_plot.iloc[-1]
        close_now = anchor_row["Close"]
        close_future = anchor_row["future_close"]
        label = anchor_row["label"]
        
        start_str = data_to_plot.index[0].strftime("%Y-%m-%d")
        end_str = data_to_plot.index[-1].strftime("%Y-%m-%d")
        filename = f"{ticker}_{start_str}_to_{end_str}_{window_index}.png"
        filepath = os.path.join(ticker_folder, filename)

        
        try:
            create_mplfinance_chart(data_to_plot, filepath, mav=window_size, xlim=None, add_rsi=USE_RSI, add_bb=USE_BB, add_ma=USE_MA, add_volume=USE_VOLUME)
            label_rows.append({
                "image_path": filepath,
                "ticker": ticker,
                "start_date": start_str,
                "end_date": end_str,
                "close_now": close_now,
                "close_future": close_future,
                "label": label
            })
        except Exception as e:
            print(f"Error processing ticker {ticker} window {window_index}: {e}")
        window_index += 1

    return label_rows

def process_ticker_wrapper(ticker_tuple):
    ticker, ticker_data = ticker_tuple
    label_rows = generate_images_for_ticker(ticker, ticker_data.copy(), WINDOW_SIZE, HORIZON, OUTPUT_DIR)
    return label_rows

##############################
#       RUN PIPELINE         #
##############################

def run_pipeline_joblib(tickers_group, n_jobs=-1):   
    """
    Process all tickers in parallel using joblib. Setting n_jobs=-1 utilizes all available CPU cores.
    """
    results = Parallel(n_jobs=n_jobs, prefer="processes")( # sæt (prefer="threads") hvis du vil have print("") virker så du kan se hvad der sker. Brug (prefer="processes") for at koden eksekvere hurtigere.
        delayed(process_ticker_wrapper)(ticker_tuple) for ticker_tuple in tqdm(tickers_group, desc="Processing tickers")
    )

    all_label_rows = []
    for ticker_rows in results:
        all_label_rows.extend(ticker_rows)

    labels_df = pd.DataFrame(all_label_rows)
    labels_df.to_csv(LABELS_CSV_PATH, index=False)

    print(f"Saved labels CSV to: {LABELS_CSV_PATH}")
    print("Image generation complete!")

# Run the pipeline using joblib with all available cores
print("Done initial data loading. Now Starting to create images")
run_pipeline_joblib(grouped, n_jobs=-1)

Finished importing Libraries
Current working directory: c:\Users\morte\Desktop\speciale-i-datavidenskab-2026
Absolute CSV path: c:\Users\morte\Desktop\speciale-i-datavidenskab-2026\CRSP_top100_latest.csv
Exists: True
OUTPUT_DIR is: vol_1_ma_1_bb_0_rsi_0
Exists? True
Processing 11 tickers from 2020-01-02 to 2024-12-31...
Done initial data loading. Now Starting to create images


Processing tickers: 100%|██████████| 11/11 [00:00<00:00, 7197.71it/s]


Saved labels CSV to: vol_1_ma_1_bb_0_rsi_0\labels.csv
Image generation complete!
